In [23]:
!pip install pyspark
!pip install findspark

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import findspark
findspark.init()

In [26]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf

In [27]:
spark=SparkSession.builder\
    .master("local[*]")\
    .appName("CardAmount")\
    .getOrCreate()

In [28]:
sc=spark.sparkContext

## Read Data

In [29]:
# data="in.csv"
# uploaded in.csv to drive/MyDrive
data = '/content/drive/MyDrive/in.csv'


In [30]:
data_rdd=sc.textFile(data)

In [31]:
# display data
# data_rdd.take(100)



In [32]:
#data count
# data_rdd.count()

In [33]:
data_rdd = data_rdd.map(lambda x: x.split(","))

In [34]:
# display data
# data_rdd.take(40)

[['224', '10'],
 ['836', '100'],
 ['81', '5'],
 ['809', '50'],
 ['786', '25'],
 ['63', '5'],
 ['478', '25'],
 ['330', '10'],
 ['511', '15'],
 ['450', '25'],
 ['521', '25'],
 ['150', '50'],
 ['671', '5'],
 ['971', '5'],
 ['439', '10'],
 ['536', '50'],
 ['253', '5'],
 ['425', '15'],
 ['638', '50'],
 ['841', '10'],
 ['324', '10'],
 ['125', '50'],
 ['937', '15'],
 ['898', '5'],
 ['461', '50'],
 ['295', '10'],
 ['683', '10'],
 ['331', '25'],
 ['187', '15'],
 ['873', '100'],
 ['303', '50'],
 ['119', '5'],
 ['589', '100'],
 ['742', '15'],
 ['81', '25'],
 ['47', '100'],
 ['521', '50'],
 ['617', '15'],
 ['228', '5'],
 ['891', '15']]

In [35]:
# transforms each row in the RDD into a key-value tuple (mapping)
data_map=data_rdd.map(lambda  line:(line[0],int(line[1])))

In [36]:
# display data
data_map.take(4)

[('224', 10), ('836', 100), ('81', 5), ('809', 50)]

In [ ]:
# group and sum all amounts by customer ID (reduce)
# NOTE: I sort using numeric value
data_count_RBK = data_map.reduceByKey(lambda x, y: x + y).sortBy(lambda x: int(x[0]))

In [38]:
# display data
data_count_RBK.take(40)

[('0', 2554710),
 ('1', 2540660),
 ('2', 2562385),
 ('3', 2545740),
 ('4', 2543400),
 ('5', 2569725),
 ('6', 2559895),
 ('7', 2563020),
 ('8', 2580355),
 ('9', 2565660),
 ('10', 2525035),
 ('11', 2561655),
 ('12', 2567315),
 ('13', 2548960),
 ('14', 2561640),
 ('15', 2544125),
 ('16', 2566590),
 ('17', 2561645),
 ('18', 2573245),
 ('19', 2564715),
 ('20', 2561125),
 ('21', 2597905),
 ('22', 2546515),
 ('23', 2566060),
 ('24', 2580670),
 ('25', 2553140),
 ('26', 2551620),
 ('27', 2577430),
 ('28', 2560910),
 ('29', 2564450),
 ('30', 2572195),
 ('31', 2572830),
 ('32', 2566775),
 ('33', 2551710),
 ('34', 2561460),
 ('35', 2572100),
 ('36', 2541460),
 ('37', 2560805),
 ('38', 2566170),
 ('39', 2527425)]

In [39]:
# sort by ID
output_data = data_count_RBK.map(lambda x: f"{x[0]}\t{x[1]}")

In [47]:
# save the output
output_data.coalesce(1).saveAsTextFile("output")